# Python GIS Ecosystem Demo: End-to-End Geospatial Workbench

This notebook provides concise, practical demos across the Python GIS stack.

It focuses on **offline-friendly examples** for reproducibility, and marks internet-dependent examples as optional.

Covered packages include: `geopandas`, `shapely`, `pyogrio`, `fiona`, `pyproj`, `rasterio`, `xarray`, `rioxarray`, `rasterstats`, `pysal`, `pyarrow`, `sqlalchemy`, `geoalchemy2`, `folium`, `ipyleaflet`, `movingpandas`, `momepy`, `pystac`, `stackstac`, `osmnx`, `OWSLib`.

In [ ]:
import warnings
import numpy as np

np.random.seed(42)
warnings.filterwarnings('ignore')

In [ ]:
import importlib.metadata as ilmd
import json

versions = {
    'geopandas': ilmd.version('geopandas'),
    'rasterio': ilmd.version('rasterio'),
    'pyproj': ilmd.version('pyproj'),
    'xarray': ilmd.version('xarray'),
}
print(json.dumps(versions, indent=2))

## 1) Vector Modeling with GeoPandas + Shapely

Create synthetic districts and observation points, then aggregate point measurements per district.

In [ ]:
import numpy as np
import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Point, Polygon

# 4 square districts centered around Zurich in Web Mercator.
# Using a real-world anchor makes later web-map demos meaningful.
center_x, center_y = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True).transform(8.5417, 47.3769)
half = 1000
district_polys = [
    Polygon([(center_x - half, center_y - half), (center_x, center_y - half), (center_x, center_y), (center_x - half, center_y)]),
    Polygon([(center_x, center_y - half), (center_x + half, center_y - half), (center_x + half, center_y), (center_x, center_y)]),
    Polygon([(center_x - half, center_y), (center_x, center_y), (center_x, center_y + half), (center_x - half, center_y + half)]),
    Polygon([(center_x, center_y), (center_x + half, center_y), (center_x + half, center_y + half), (center_x, center_y + half)]),
]
districts = gpd.GeoDataFrame(
    {'district': ['A', 'B', 'C', 'D']}, geometry=district_polys, crs='EPSG:3857'
)

# 150 random sensor points with synthetic PM2.5 signal
xy = np.column_stack([
    np.random.uniform(center_x - half, center_x + half, size=150),
    np.random.uniform(center_y - half, center_y + half, size=150),
])
pts = [Point(x, y) for x, y in xy]
obs = gpd.GeoDataFrame(
    {'pm25': np.random.gamma(shape=2.0, scale=8.0, size=150)}, geometry=pts, crs='EPSG:3857'
)

joined = gpd.sjoin(obs, districts, predicate='within', how='left')
district_stats = joined.groupby('district', dropna=True).agg(pm25_mean=('pm25', 'mean'), n=('pm25', 'size')).reset_index()
districts = districts.merge(district_stats, on='district', how='left')
districts

In [ ]:
ax = districts.plot(column='pm25_mean', cmap='viridis', edgecolor='black', legend=True, figsize=(6, 6))
obs.plot(ax=ax, color='white', edgecolor='black', markersize=10, alpha=0.6)
ax.set_title('District PM2.5 mean and monitoring points')
ax.set_axis_off()

## 2) Geometry Operations and Morphology (Shapely)

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union

# Union districts and build a 150 m boundary buffer ring
city = unary_union(districts.geometry)
outer = city.buffer(150)
ring = outer.difference(city)

geom_demo = gpd.GeoDataFrame(
    {'layer': ['city', 'buffer_ring'], 'area_m2': [city.area, ring.area]},
    geometry=[city, ring],
    crs=districts.crs,
)
geom_demo

## 3) CRS and Coordinate Transformation (pyproj)

In [ ]:
import pyproj

# Transform Zurich coordinates from WGS84 to Swiss LV95
lon, lat = 8.5417, 47.3769
transformer = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:2056', always_xy=True)
e, n = transformer.transform(lon, lat)
print(f'WGS84: ({lon:.4f}, {lat:.4f}) -> LV95: ({e:.1f}, {n:.1f})')

## 4) Vector I/O Engines: pyogrio, fiona, and GDAL bindings

In [ ]:
from pathlib import Path
import geopandas as gpd
from osgeo import ogr

tmp_dir = Path('/tmp/python_gis_demo')
tmp_dir.mkdir(parents=True, exist_ok=True)
gpkg = tmp_dir / 'districts.gpkg'

districts.to_file(gpkg, layer='districts', driver='GPKG', engine='pyogrio')
districts_fiona = gpd.read_file(gpkg, layer='districts', engine='fiona')
print('rows read with fiona:', len(districts_fiona))

ds = ogr.Open(str(gpkg))
print('layers via GDAL/OGR:', [ds.GetLayerByIndex(i).GetName() for i in range(ds.GetLayerCount())])

## 5) Raster Synthesis and Analysis (rasterio + numpy)

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.io import MemoryFile

width = height = 100
pixel = 20
transform = from_origin(0, 2000, pixel, pixel)

x = np.linspace(-2, 2, width)
y = np.linspace(-2, 2, height)
xx, yy = np.meshgrid(x, y)
elev = (np.exp(-(xx**2 + yy**2)) * 1200 + (xx + 2) * 100).astype('float32')

memfile = MemoryFile()
with memfile.open(
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype='float32',
    crs='EPSG:3857',
    transform=transform,
) as ds:
    ds.write(elev, 1)

with memfile.open() as ds:
    arr = ds.read(1)
    gy, gx = np.gradient(arr, pixel, pixel)
    slope = np.sqrt(gx**2 + gy**2)

arr.min(), arr.max(), slope.mean()

## 6) xarray + rioxarray: Raster Metadata, Clip, and Reproject

In [ ]:
import rioxarray

with memfile.open() as ds:
    da = rioxarray.open_rasterio(ds).squeeze(drop=True)

# Clip to district A footprint
clip_geom = [districts.loc[districts['district'] == 'A', 'geometry'].iloc[0]]
da_clip = da.rio.clip(clip_geom, districts.crs)
da_4326 = da_clip.rio.reproject('EPSG:4326')

print('original shape:', tuple(da.shape), 'clipped:', tuple(da_clip.shape), 'reprojected:', tuple(da_4326.shape))

## 7) Vector-Raster Bridge: Zonal Statistics (rasterstats)

In [ ]:
import pandas as pd
from rasterstats import zonal_stats

zs = zonal_stats(
    vectors=districts.geometry,
    raster=elev,
    affine=transform,
    stats=['mean', 'min', 'max', 'std'],
)
zonal_df = pd.DataFrame(zs)
districts_zonal = pd.concat([districts[['district']].reset_index(drop=True), zonal_df], axis=1)
districts_zonal

## 8) Spatial Autocorrelation (PySAL)

In [ ]:
import numpy as np
from libpysal.weights import Queen
from esda.moran import Moran

w = Queen.from_dataframe(districts, use_index=False)
w.transform = 'r'
y = districts['pm25_mean'].fillna(districts['pm25_mean'].mean()).to_numpy()
mi = Moran(y, w)
print({'I': float(mi.I), 'p_sim': float(mi.p_sim)})

## 9) Geospatial Storage: GeoParquet and Arrow

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import geopandas as gpd

parquet_path = tmp_dir / 'districts.parquet'
districts.to_parquet(parquet_path, index=False)
districts_back = gpd.read_parquet(parquet_path)

# Arrow table for a tabular projection without geometry
attrs_tbl = pa.Table.from_pandas(districts.drop(columns='geometry'))
pq.write_table(attrs_tbl, tmp_dir / 'district_attrs.parquet')

print('GeoParquet rows:', len(districts_back), 'columns:', districts_back.columns.tolist())

## 10) SQL Geometry Typing with SQLAlchemy + GeoAlchemy2

In [ ]:
from sqlalchemy import Table, Column, Integer, MetaData, select
from geoalchemy2 import Geometry

# Build SQL metadata without connecting to a DB (safe offline demo).
meta = MetaData()
tbl = Table(
    'district_measurements',
    meta,
    Column('id', Integer, primary_key=True),
    Column('geom', Geometry('POLYGON', srid=3857)),
    Column('pm25_mean', Integer),
)
stmt = select(tbl.c.id, tbl.c.pm25_mean).where(tbl.c.pm25_mean > 10)
print(stmt)

## 11) Interactive Mapping: Folium and ipyleaflet

In [ ]:
import folium

districts_wgs84 = districts.to_crs(4326)
bounds = districts_wgs84.total_bounds  # minx, miny, maxx, maxy
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
m = folium.Map(location=center, zoom_start=13, tiles='cartodbpositron')
folium.GeoJson(
    districts_wgs84.to_json(),
    name='districts',
    tooltip=folium.GeoJsonTooltip(fields=['district', 'pm25_mean', 'n']),
).add_to(m)
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
m

In [ ]:
from ipyleaflet import Map, GeoData

bounds = districts_wgs84.total_bounds
center = ((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2)
m2 = Map(center=center, zoom=13)
m2.add_layer(GeoData(geo_dataframe=districts_wgs84, style={'color': 'black', 'fillOpacity': 0.3}))
m2

## 12) Trajectories and Urban Morphology (movingpandas + momepy)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import movingpandas as mpd
import momepy
from shapely.geometry import LineString

# Synthetic moving object trajectory
t = pd.date_range('2025-01-01', periods=20, freq='5min')
traj_df = pd.DataFrame({
    'id': 1,
    't': t,
    'x': np.linspace(100, 1800, len(t)) + np.random.normal(0, 30, len(t)),
    'y': np.linspace(200, 1600, len(t)) + np.random.normal(0, 30, len(t)),
})
traj_gdf = gpd.GeoDataFrame(traj_df, geometry=gpd.points_from_xy(traj_df.x, traj_df.y), crs='EPSG:3857')
traj_collection = mpd.TrajectoryCollection(traj_gdf, traj_id_col='id', t='t')
traj = traj_collection.trajectories[0]
print('trajectory length (m):', round(traj.get_length(), 2))

# Minimal momepy demo: convert synthetic street centerlines to network graph
streets = gpd.GeoDataFrame(
    geometry=[
        LineString([(0, 500), (2000, 500)]),
        LineString([(0, 1500), (2000, 1500)]),
        LineString([(500, 0), (500, 2000)]),
        LineString([(1500, 0), (1500, 2000)]),
    ],
    crs='EPSG:3857',
)
G = momepy.gdf_to_nx(streets, approach='primal')
print('street graph nodes/edges:', G.number_of_nodes(), G.number_of_edges())

## 13) STAC and Optional Cloud/API Integrations

In [ ]:
import pandas as pd
import pystac

# Offline STAC object construction
item = pystac.Item(
    id='synthetic-scene-001',
    geometry={
        'type': 'Polygon',
        'coordinates': [[[8.5, 47.3], [8.6, 47.3], [8.6, 47.4], [8.5, 47.4], [8.5, 47.3]]],
    },
    bbox=[8.5, 47.3, 8.6, 47.4],
    datetime=pd.Timestamp('2025-01-01T10:00:00Z').to_pydatetime(),
    properties={'eo:cloud_cover': 7.2},
)
print('STAC item id:', item.id)

# Optional internet-dependent demos
try:
    import pystac_client
    client = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1')
    print('STAC API reachable:', client.id)
except Exception as exc:
    print('STAC API demo skipped (offline or blocked):', type(exc).__name__)

try:
    import osmnx as ox
    G_drive = ox.graph_from_place('Zurich, Switzerland', network_type='drive', simplify=True)
    print('OSMnx graph nodes:', len(G_drive.nodes))
except Exception as exc:
    print('OSMnx demo skipped (offline or blocked):', type(exc).__name__)

try:
    from owslib.wms import WebMapService
    wms = WebMapService('https://ahocevar.com/geoserver/wms', version='1.3.0')
    print('WMS title:', wms.identification.title)
except Exception as exc:
    print('OWSLib demo skipped (offline or blocked):', type(exc).__name__)

## 14) Takeaways

- `GeoPandas + Shapely` handles vector ETL and geometry analytics elegantly.
- `Rasterio + rioxarray + xarray` covers raster IO, metadata-aware transformations, and N-D workflows.
- `rasterstats + PySAL` bridge classic GIS overlays with spatial statistics.
- `pyarrow/GeoParquet` improves interoperability and analytics-readiness.
- `folium/ipyleaflet` provide rapid interactive map communication in notebooks.
- Optional web ecosystem tools (`pystac-client`, `osmnx`, `OWSLib`) unlock cloud and live-service integration when network is available.